In [ ]:
// ==== Setup ====

import java.util.Random;

int comparisons, swaps;   // reset before each run

# Sorting: quadratic sorts and merge sort

## Where sorted data comes from

Binary search needed a sorted array. Nothing sorts itself.

Sort once, search many times - the same trade from last lecture, now on the other side of the ledger.

## Counting, again

One allowed move: **compare** two elements, **swap** two elements.

Count both, as a function of N - same convention as searching.

## Bubble sort

Compare neighbors, left to right. Swap if out of order. Repeat the pass.

Each full pass carries the largest remaining element to its final spot.

In [ ]:
void bubbleSort(double[] list) {
    double tmp;
    for (int i = 0; i < list.length - 1; i++) {
        for (int j = 0; j < list.length - i - 1; j++) {
            comparisons++;
            if (list[j] > list[j + 1]) {
                tmp = list[j];
                list[j] = list[j + 1];
                list[j + 1] = tmp;
                swaps++;
            }
        }
    }
}

double[] example = {13, 6, 2, 7, 11, 4, 8, 15, 9, 3};

double[] bub = example.clone();
comparisons = 0; swaps = 0;
bubbleSort(bub);
System.out.println(java.util.Arrays.toString(bub));
System.out.println("comparisons=" + comparisons + "  swaps=" + swaps);

`i` counts passes. The inner bound `length - i - 1` shrinks each pass: the tail is already in place, no need to recheck it.

## Selection sort

Find the smallest remaining element. Swap it to the front. Repeat on what's left.

In [ ]:
void selectionSort(double[] list) {
    for (int i = 0; i < list.length - 1; i++) {
        double currentMin = list[i];
        int currentMinIndex = i;
        for (int j = i + 1; j < list.length; j++) {
            comparisons++;
            if (currentMin > list[j]) {
                currentMin = list[j];
                currentMinIndex = j;
            }
        }
        if (currentMinIndex != i) {
            list[currentMinIndex] = list[i];   // no temp needed: currentMin already holds it
            list[i] = currentMin;
            swaps++;
        }
    }
}

double[] sel = example.clone();
comparisons = 0; swaps = 0;
selectionSort(sel);
System.out.println(java.util.Arrays.toString(sel));
System.out.println("comparisons=" + comparisons + "  swaps=" + swaps);

Same comparisons as bubble sort - `n(n-1)/2`, independent of how out of order the data is.

Far fewer swaps: **at most one per pass**.

When would that difference matter?

## Insertion sort

Split the array into a sorted part and an unsorted part - initially just the first element is sorted.

Take the next unsorted element, shift larger elements right, drop it into place.

In [ ]:
void insertionSort(double[] list) {
    for (int i = 1; i < list.length; i++) {
        double currentElement = list[i];
        int k;
        for (k = i - 1; k >= 0; k--) {
            comparisons++;
            if (list[k] <= currentElement) break;
            list[k + 1] = list[k];
            swaps++;
        }
        list[k + 1] = currentElement;
    }
}

double[] ins = example.clone();
comparisons = 0; swaps = 0;
insertionSort(ins);
System.out.println(java.util.Arrays.toString(ins));
System.out.println("comparisons=" + comparisons + "  swaps=" + swaps);

In [ ]:
double[] sorted = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10};
comparisons = 0; swaps = 0;
insertionSort(sorted);
System.out.println("already sorted : comparisons=" + comparisons + "  swaps=" + swaps);

double[] reversed = {10, 9, 8, 7, 6, 5, 4, 3, 2, 1};
comparisons = 0; swaps = 0;
insertionSort(reversed);
System.out.println("reverse sorted : comparisons=" + comparisons + "  swaps=" + swaps);

N-1 comparisons on sorted input: **O(N)**, not O(N^2).

The only one of the three whose cost depends on how sorted the input already is.

Does that ever matter in practice?

## Growth, measured

Same doubling test as always: random arrays, increasing N.

In [ ]:
Random rng = new Random(42);
double[] randomArray(int n) {
    double[] a = new double[n];
    for (int i = 0; i < n; i++) a[i] = rng.nextDouble() * 1_000_000;
    return a;
}

System.out.printf("%8s %12s %12s %12s%n", "N", "bubble", "selection", "insertion");
double pb = 0, ps = 0, pi = 0;
for (int size = 500; size <= 8000; size *= 2) {
    comparisons = 0; bubbleSort(randomArray(size));    double cb = comparisons;
    comparisons = 0; selectionSort(randomArray(size)); double cs = comparisons;
    comparisons = 0; insertionSort(randomArray(size)); double ci = comparisons;
    System.out.printf("%8d %12.0f %12.0f %12.0f%n", size, cb, cs, ci);
    if (pb != 0) System.out.printf("%8s %12.2f %12.2f %12.2f%n", "", cb / pb, cs / ps, ci / pi);
    pb = cb; ps = cs; pi = ci;
}

All three ratio to **4**: doubling N quadruples the work. Same shape, different constants.

---
# Merge sort
Let's solve sorting using last lecture's recipe:
1. **Base case**: What instances of the problem are trivial to solve?
2. **Recursive call**: Given a generic instance, how can I make it simpler, or "closer" to one of the trivial instances?
3. **Combine the results**: How can I use the answer of the simpler instances to solve the harder one?
4. **Check for termination**: Does every path in my algorithm eventually reach a base case?

If we sort using a quadratic sort after one split, how many operations is that?

after 1 and 2:
```
mergeSort(array):
    if array.length <= 1: return array    // 1. base case
    left  = mergeSort(firstHalf(array))   // 2. shrink
    right = mergeSort(secondHalf(array))  // 2. shrink
    ???                                   // 3. combine - what goes here?
```

Both halves come back **sorted** - that is what the recursive calls guarantee. Combine two sorted arrays into one new sorted array: walk both from the front, always take the smaller head.

What is the complexity of this operation, in the combined size n?

In [ ]:
double[] merge(double[] x, double[] y) {
    double[] result = new double[x.length + y.length];
    int i = 0, j = 0, k = 0;

    while (i < x.length && j < y.length) {
        comparisons++;
        if (x[i] <= y[j]) result[k++] = x[i++];
        else               result[k++] = y[j++];
    }
    while (i < x.length) result[k++] = x[i++];   // x has leftovers
    while (j < y.length) result[k++] = y[j++];   // y has leftovers

    return result;
}

Putting it all together:

In [ ]:
double[] mergeSort(double[] array) {
    if (array.length <= 1) return array;        // base case
    int mid = array.length / 2;
    double[] left  = mergeSort(java.util.Arrays.copyOfRange(array, 0, mid));
    double[] right = mergeSort(java.util.Arrays.copyOfRange(array, mid, array.length));
    return merge(left, right);
}

comparisons = 0;
double[] m = mergeSort(example);
System.out.println("original : " + java.util.Arrays.toString(example));
System.out.println("sorted   : " + java.util.Arrays.toString(m));
System.out.println("comparisons=" + comparisons);

Does every path reach the base case?

**How many levels of recursion?**

## Tracing the recursion

Same `example` array. Indent by depth, print on the way down and on the way back up - same device as `factorialTraced` last lecture.

In [ ]:
double[] mergeSortTraced(double[] array, int depth) {
    String pad = "  ".repeat(depth);
    System.out.println(pad + "-> " + java.util.Arrays.toString(array));
    if (array.length <= 1) { System.out.println(pad + "<- base case"); return array; }

    int mid = array.length / 2;
    double[] left  = mergeSortTraced(java.util.Arrays.copyOfRange(array, 0, mid), depth + 1);
    double[] right = mergeSortTraced(java.util.Arrays.copyOfRange(array, mid, array.length), depth + 1);
    double[] merged = merge(left, right);

    System.out.println(pad + "<- merged   " + java.util.Arrays.toString(merged));
    return merged;
}

mergeSortTraced(example, 0);

How many levels deep does the trace above go? Same halving argument as binary search last lecture: `N -> N/2 -> N/4 -> ... -> 1`. Does it match `log2 N`?

### How much work per level?

At any one level, the pieces being merged are disjoint slices of the original array - together, they still add up to n elements. One pass over n elements in total, however many separate arrays that work is split across.

`O(n)` per level * `log2 N` levels = **O(n log n)**.

Which part does the "actual work"?

## Sorting in place

The version above copies arrays at every single call. Same algorithm, rewritten to reuse the original array - index ranges instead of new arrays, one small buffer per merge instead of permanent copies.

Same names, `merge` and `mergeSort` - Java tells the two versions apart by parameter list.

In [ ]:
void merge(double[] array, int first, int mid, int last) {
    double[] temp = new double[last - first + 1];
    int i = first, j = mid + 1, k = 0;

    while (i <= mid && j <= last) {
        comparisons++;
        if (array[i] <= array[j]) temp[k++] = array[i++];
        else                      temp[k++] = array[j++];
    }
    while (i <= mid) temp[k++] = array[i++];   // left has leftovers
    while (j <= last) temp[k++] = array[j++];  // right has leftovers

    for (int t = 0; t < temp.length; t++) array[first + t] = temp[t];
}

void mergeSort(double[] array, int first, int last) {
    if (first >= last) return;                 // base case: 0 or 1 elements
    int mid = (first + last) / 2;
    mergeSort(array, first, mid);
    mergeSort(array, mid + 1, last);
    merge(array, first, mid, last);
}

double[] ip = example.clone();   // clone: this version sorts in place
comparisons = 0;
mergeSort(ip, 0, ip.length - 1);
System.out.println(java.util.Arrays.toString(ip));
System.out.println("comparisons=" + comparisons);
System.out.println("example untouched: " + java.util.Arrays.toString(example));

## Extra space

Every call to `merge` allocates a new array, sized to the range it is merging. The recursion has `log n` levels.

Is the extra space O(n), or O(n log n)? (Hint: `merge` never overlaps with itself - by the time a call runs its own `merge`, both of its recursive calls have already returned.)

## A closer look at mergesort

```
1  mergeSort( array )
2    mergeSortRec (array, 0, array.length-1)
```

```
3  mergeSortRec ( array, firstIndex, lastIndex )
4    if (firstIndex >= lastIndex)
5      return
6    mid = (firstIndex+lastIndex)/2
7    mergeSortRec(array, firstIndex, mid)
8    mergeSortRec(array, mid+1, lastIndex)
9    merge( array, firstIndex, mid, mid+1, lastIndex)
```

- What does the array look like once the call on line 7 completely finishes (every recursive call inside it included)?
- What does the array look like once the call on line 8 completely finishes?
- Line 2: what if the third argument were `array.length` instead of `array.length-1`?
- Line 8: what if the second argument were `mid` instead of `mid+1`?
- Line 7: what if the third argument were `mid-1` instead of `mid`?